In [ ]:
import cv2
import numpy as np
import os

# =========================================================
# CONFIGURACIÓN
# =========================================================

outputXSize = 800
latRate = 1/4 # Porcentaje de la imagen horizontal que son paredes. Actualmente 1/4 de la imagen son paredes.
nTilesInVArea = 5.5

# =========================================================
# Homografía
# =========================================================

latMargin = outputXSize * latRate / 2
outputYSize = int((outputXSize - (2 * latMargin)) * nTilesInVArea)
WIDTH, HEIGHT = outputXSize, outputYSize

imgs=["Perspective", "Test1"]
# Cargar imagen
input_path = f"imagenes/{imgs[1]}.png"

img = cv2.imread(input_path)

if img is None:
    print("No se pudo cargar la imagen.")
    exit()

Base_W = 3280 
Base_H = 2464

h, w = img.shape[:2]

scale_x = w / Base_W
scale_y = h / Base_H

# Los 4 puntos del trapecio del suelo
pts_src = np.array([
    [322,  2463],   # esquina inferior izquierda
    [2956, 2463],  # esquina inferior derecha
    [1848,  814],  # superior derecha
    [1430,  814]   # superior izquierda
], dtype=np.float32)

pts_src = pts_src.copy()
pts_src[:, 0] *= scale_x  # Escala la coordenada X
pts_src[:, 1] *= scale_y  # Escala la coordenada Y

pts_roi = pts_src.astype(np.int32)

# Vista cenital destino
pts_dst = np.array([
    [latMargin, HEIGHT],
    [WIDTH - latMargin, HEIGHT],
    [WIDTH - latMargin, 0],
    [latMargin, 0]
], dtype=np.float32)

# Calcula el movimiento
H = cv2.getPerspectiveTransform(pts_src, pts_dst)

# Genera el movimiento
top_down = cv2.warpPerspective(img,H,(WIDTH, HEIGHT))

# =========================================================
# GUARDAR IMAGEN CENITAL
# =========================================================

# Crear carpeta si no existe
os.makedirs("imagenesCenitales", exist_ok=True)

# Obtener nombre del archivo sin extensión
nombre = os.path.splitext(os.path.basename(input_path))[0]

# Obtener extensión original
extension = os.path.splitext(input_path)[1]

# Crear nuevo nombre
output_path = f"imagenesCenitales/{nombre}_cenital{extension}"

# Guardar imagen
cv2.imwrite(output_path, top_down)
print()

In [16]:
# =========================================================
# VISUALIZACIÓN
# =========================================================

# Dibujar puntos usados
img_check = img.copy()

for i, pt in enumerate(pts_src):
    cv2.circle(
        img_check,
        tuple(pt.astype(int)),
        7,
        (0, 255, 0),
        -1
    )

    cv2.putText(
        img_check,
        str(i + 1),
        tuple(pt.astype(int)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 0, 255),
        2
    )

# Dibujar trapecio
cv2.polylines(
    img_check,
    [pts_src.astype(np.int32)],
    True,
    (255, 0, 0),
    2
)

#cv2.imshow("Puntos Usados Homografia", img_check)
#cv2.imshow("Vista Cenital", top_down)
print()

In [17]:
#Tratamiento de imagen
output_path = f"imagenesCenitales/{nombre}_cenitalBW{extension}"

processed = cv2.cvtColor(top_down, cv2.COLOR_BGR2HSV)

lower_brown = np.array([10, 40, 20])
upper_brown = np.array([30, 255, 255])
mask = cv2.inRange(processed, lower_brown, upper_brown)

kernel = np.ones((7,7), np.uint8)
mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

#cv2.imshow("Vista Cenital", mask)

cv2.waitKey(0)
cv2.destroyAllWindows()

cv2.imwrite(output_path, mask)
print()